# Retail Inventory Planning

A guided analysis of retail sales, forecasting, and illustrative reorder policies. Read the README and `docs/learning_guide.md` alongside this notebook.

**Setup:** download the UCI spreadsheet with `python download_data.py`. In Colab, upload `analysis.py`, `report.py`, and `Online Retail.xlsx` to the runtime. Set the path in the cell below. Lead times, costs, and inventory states are hypothetical.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from analysis import run

input_file = Path("data/Online Retail.xlsx")
# In Colab: input_file = Path("/content/Online Retail.xlsx")
results = run(input_file, Path("results"))


## 1. Audit and ABC segmentation

There are 3,709 eligible SKUs across Jan–Nov, but only 3,538 with training-period sales for the Jan–Sep ABC analysis. Segmentation uses selling revenue, not inventory value or profit. **Your task:** explain why cancellations are excluded and why zero recorded sales could hide unmet demand.

In [ ]:
print(results["audit"])
display(results["abc"].head(20))


## 2. Forecasts on later dates

The four prespecified benchmarks predict one day ahead using only earlier sales. WAPE pools absolute error across SKU-days and divides by total observed sales. These forecasts are not directly used by the fixed-parameter inventory simulation.

In [ ]:
display(results["accuracy"])
sku = results["abc"].iloc[0]["StockCode"]
one = results["forecasts"].query("StockCode == @sku").set_index("Date")
one[["units", "forecast_28d", "weekday_mean_4w"]].plot(figsize=(10, 4), title=f"Holdout sales and forecasts: SKU {sku}")
plt.ylabel("Units per day")
plt.show()


## 3. Reorder policies

Policy inputs use Jan–Aug, followed by a 30-day September warm-up. For each SKU and lead-time scenario, both policies start with the same stock. Reported metrics cover Oct–Nov only. The normal-theory quantile 1.645 does not guarantee 95% unit fill rate.

In [ ]:
display(results["summary"])
display(results["policies"].head(8))


## 4. Lead-time sensitivity

Compare three assumed supplier lead times. **Your task:** explain why higher simulated service is useful but does not by itself establish that a policy is cheaper or optimal.

In [ ]:
display(results["sensitivity"])
import runpy
from IPython.display import display, Image
runpy.run_path("report.py", run_name="__main__")
display(Image(filename="results/lead_time_sensitivity.png"))


## 5. Write your recommendation

Write three sentences in the cell below: your business recommendation, its quantitative support, and the missing information needed before implementation. Do not claim realized cost savings or retailer stockout reductions.

**My recommendation:**

**Supporting result:**

**Data I would request next:**